# STG_7 — Optimización de hiperparámetros del modelo ganador (LightGBM)

Toma el modelo ganador de STG_6 (LGBMClassifier) y busca mejores hiperparámetros
con `RandomizedSearchCV`, usando el **mismo** `TimeSeriesSplit` de STG_6.

Regla obligatoria: la búsqueda usa `cv=TimeSeriesSplit`, nunca CV aleatoria.


In [ ]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import f1_score
from lightgbm import LGBMClassifier

DATA_DIR  = Path("../data")
SPEC_DIR  = Path("../specs/009-modelado-prediccion-votos")
RANDOM_STATE = 42


## T20 — Carga y reconstrucción del split de STG_6

In [ ]:
# T20 — Misma carga y split que STG_6 (reproducibilidad garantizada)
df = pd.read_csv(DATA_DIR / "df_entrenamiento.csv", parse_dates=["fecha_votacion"])
df = df.sort_values("fecha_votacion").reset_index(drop=True)

META     = ["diputado", "titulo_base", "fecha_votacion", "bloque", "provincia", "tema_label"]
TARGET   = "voto"
FEATURES = [c for c in df.columns if c not in META + [TARGET]]

le_voto = LabelEncoder()
df["voto_enc"] = le_voto.fit_transform(df[TARGET])

# Split identico al de STG_6
corte_idx = int(len(df) * 0.80)
df_train  = df.iloc[:corte_idx].copy()
df_hold   = df.iloc[corte_idx:].copy()
X_train   = df_train[FEATURES].values
y_train   = df_train["voto_enc"].values
X_hold    = df_hold[FEATURES].values
y_hold    = df_hold["voto_enc"].values

# TimeSeriesSplit identico al de STG_6
tscv = TimeSeriesSplit(n_splits=5)

fecha_corte = df.iloc[corte_idx]["fecha_votacion"]
print(f"Train  : {len(df_train):,} filas hasta {df_train['fecha_votacion'].max().date()}")
print(f"Holdout: {len(df_hold):,} filas desde {df_hold['fecha_votacion'].min().date()}")
print(f"Features: {len(FEATURES)}")

# Verificar mismo split que STG_6
assert df_train["fecha_votacion"].max() <= df_hold["fecha_votacion"].min()
print()
print("T20 PASA: split identico a STG_6 verificado.")


## T21 — Búsqueda de hiperparámetros con RandomizedSearchCV

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
import scipy.stats as stats

# Modelo base con los parametros que ya funcionaron en STG_6
modelo_base = LGBMClassifier(
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=-1
)

# Grilla de hiperparametros a explorar
param_dist = {
    'n_estimators'     : [100, 200, 300, 400, 500, 600],
    'learning_rate'    : [0.01, 0.03, 0.05, 0.08, 0.10, 0.15],
    'num_leaves'       : [31, 47, 63, 80, 100, 127],
    'min_child_samples': [10, 20, 30, 50, 80],
    'colsample_bytree' : [0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    'subsample'        : [0.6, 0.7, 0.8, 0.9, 1.0],
}

# RandomizedSearchCV con el mismo tscv de T20 — nunca split aleatorio
buscador = RandomizedSearchCV(
    estimator   = modelo_base,
    param_distributions = param_dist,
    n_iter      = 30,
    cv          = tscv,
    scoring     = 'f1_macro',
    random_state= RANDOM_STATE,
    n_jobs      = 1,       # n_jobs=1 para evitar doble paralelismo con LightGBM
    verbose     = 1,
    refit       = True,
)

print("Iniciando busqueda de hiperparametros (30 iteraciones x 5 folds = 150 ajustes)...")
print("Esto puede tardar varios minutos.")
buscador.fit(X_train, y_train)

print()
print("Mejores hiperparametros encontrados:")
for k, v in buscador.best_params_.items():
    print(f"  {k:<22}: {v}")
print(f"  CV F1-macro (mejor)  : {buscador.best_score_:.4f}")


## T22 — Comparar modelo por defecto vs. afinado

In [ ]:
from sklearn.model_selection import cross_val_score

# Modelo afinado (ya entrenado sobre todo el train por refit=True)
modelo_afinado = buscador.best_estimator_

# Modelo por defecto de STG_6 (los parametros que ganaron la comparacion)
modelo_default = LGBMClassifier(
    n_estimators=300, learning_rate=0.05, num_leaves=63,
    class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1, verbosity=-1
)

# CV del modelo default (para comparacion justa)
cv_default = cross_val_score(modelo_default, X_train, y_train,
                             cv=tscv, scoring='f1_macro', n_jobs=1).mean()

# Holdout de ambos
modelo_default.fit(X_train, y_train)
hold_default = f1_score(y_hold, modelo_default.predict(X_hold), average='macro')
hold_afinado = f1_score(y_hold, modelo_afinado.predict(X_hold), average='macro')

# Tabla comparativa
comparacion = pd.DataFrame([
    {'Modelo': 'LightGBM por defecto (STG_6)',
     'CV F1-macro': round(cv_default, 4),
     'Holdout F1-macro': round(hold_default, 4)},
    {'Modelo': 'LightGBM afinado (STG_7)',
     'CV F1-macro': round(buscador.best_score_, 4),
     'Holdout F1-macro': round(hold_afinado, 4)},
])
print(comparacion.to_string(index=False))
print()
mejora_hold = (hold_afinado - hold_default) / hold_default * 100
print(f"Mejora en holdout: {mejora_hold:+.1f}%")

# Guardar tabla
comparacion.to_csv(SPEC_DIR / 'comparacion_default_vs_afinado.csv', index=False)
print("Guardado: comparacion_default_vs_afinado.csv")

# Grafico comparativo
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
x = ['Por defecto\n(STG_6)', 'Afinado\n(STG_7)']
cv_vals   = [cv_default, buscador.best_score_]
hold_vals = [hold_default, hold_afinado]
xi = range(len(x))
ancho = 0.35
b1 = ax.bar([i - ancho/2 for i in xi], cv_vals,   ancho, label='CV F1-macro',      color='#5B9BD5')
b2 = ax.bar([i + ancho/2 for i in xi], hold_vals,  ancho, label='Holdout F1-macro', color='#ED7D31')
ax.set_xticks(list(xi)); ax.set_xticklabels(x)
ax.set_ylim(0, 0.6); ax.set_ylabel('F1-macro')
ax.set_title('LightGBM: por defecto vs. afinado', fontsize=12, fontweight='bold')
ax.legend()
for b in list(b1) + list(b2):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.008,
            f'{b.get_height():.3f}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig(SPEC_DIR / 'comparacion_tuning.png', dpi=150, bbox_inches='tight')
plt.close()
print("Guardado: comparacion_tuning.png")
print()
print("T22 PASA.")
